# Интерпретация модели оттока клиентов

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import PartialDependenceDisplay
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    import shap
except ImportError:
    shap = None

try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError:
    LimeTabularExplainer = None

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

if shap is None or LimeTabularExplainer is None:
    print("Если пакеты не установлены, выполните: %pip install shap lime")

In [ ]:
def parse_dt(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, utc=True, errors="coerce", format="mixed")


def safe_mode(series: pd.Series):
    mode = series.mode(dropna=True)
    if not mode.empty:
        return mode.iat[0]
    non_null = series.dropna()
    return non_null.iat[0] if len(non_null) else np.nan


LOOKBACK_DAYS = 365
CHURN_HORIZON_DAYS = 180
CUTOFF_DATES = pd.to_datetime([
    "2024-06-01",
    "2024-09-01",
    "2024-12-01",
    "2025-03-01",
    "2025-06-01",
])


def build_slice(cutoff_date: pd.Timestamp, orders: pd.DataFrame, data: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    lookback_start = cutoff_date - pd.Timedelta(days=LOOKBACK_DAYS)
    horizon_end = cutoff_date + pd.Timedelta(days=CHURN_HORIZON_DAYS)

    history_orders = orders.loc[orders["created_at_naive"] < cutoff_date].copy()
    future_orders = orders.loc[
        (orders["created_at_naive"] >= cutoff_date) &
        (orders["created_at_naive"] < horizon_end)
    ].copy()

    user_history = (
        history_orders.groupby("user_id", as_index=False)
        .agg(
            total_orders_before_cutoff=("order_id", "nunique"),
            loyal_flag=("is_loyal", "max"),
            first_order_ever=("created_at_naive", "min"),
            last_order_ever=("created_at_naive", "max"),
        )
    )
    user_history = user_history[(user_history["total_orders_before_cutoff"] >= 2) | (user_history["loyal_flag"] == True)].copy()

    future_users = future_orders.groupby("user_id", as_index=False).agg(future_orders=("order_id", "nunique"))
    user_history = user_history.merge(future_users, on="user_id", how="left")
    user_history["future_orders"] = user_history["future_orders"].fillna(0)
    user_history["churn_target"] = (user_history["future_orders"] == 0).astype(int)

    data_window = data.loc[
        (data["created_at_naive"] >= lookback_start) &
        (data["created_at_naive"] < cutoff_date)
    ].copy()
    orders_window = history_orders.loc[
        (history_orders["created_at_naive"] >= lookback_start) &
        (history_orders["created_at_naive"] < cutoff_date)
    ].copy()
    events_window = events.loc[
        (events["created_at_naive"] >= lookback_start) &
        (events["created_at_naive"] < cutoff_date)
    ].copy()

    customer_orders = (
        orders_window.groupby("user_id", as_index=False)
        .agg(
            orders_last_365d=("order_id", "nunique"),
            revenue_last_365d=("order_value", "sum"),
            avg_order_value_last_365d=("order_value", "mean"),
            avg_items_per_order_last_365d=("items_count", "mean"),
            first_order_in_window=("created_at_naive", "min"),
            last_order_in_window=("created_at_naive", "max"),
        )
    )
    customer_orders["days_since_last_order"] = (cutoff_date - customer_orders["last_order_in_window"]).dt.days
    customer_orders["window_span_days"] = (customer_orders["last_order_in_window"] - customer_orders["first_order_in_window"]).dt.days

    customer_items = (
        data_window.groupby("user_id", as_index=False)
        .agg(
            return_rate_items=("is_returned", "mean"),
            cancelled_rate_items=("is_cancelled", "mean"),
            delivered_rate_items=("is_delivered", "mean"),
            avg_delivery_days=("delivery_days", "mean"),
            avg_ship_delay_days=("ship_delay_days", "mean"),
            categories_count=("category", "nunique"),
            brands_count=("brand", "nunique"),
            main_category=("category", lambda s: safe_mode(s)),
            main_department=("department", lambda s: safe_mode(s)),
            main_brand=("brand", lambda s: safe_mode(s)),
            gender=("gender", "first"),
            age=("age", "median"),
            state=("state", lambda s: safe_mode(s)),
            traffic_source=("traffic_source", lambda s: safe_mode(s)),
        )
    )

    customer_events = (
        events_window.groupby("user_id", as_index=False)
        .agg(
            sessions_last_365d=("session_id", "nunique"),
            events_last_365d=("id", "count"),
            product_views_last_365d=("is_product", "sum"),
            cart_events_last_365d=("is_cart", "sum"),
            purchase_events_last_365d=("is_purchase", "sum"),
            last_event_at=("created_at_naive", "max"),
        )
    )
    customer_events["days_since_last_event"] = (cutoff_date - customer_events["last_event_at"]).dt.days
    customer_events["events_per_session"] = customer_events["events_last_365d"] / customer_events["sessions_last_365d"]
    customer_events["cart_to_view_rate"] = customer_events["cart_events_last_365d"] / customer_events["product_views_last_365d"].replace(0, np.nan)
    customer_events["purchase_to_cart_rate"] = customer_events["purchase_events_last_365d"] / customer_events["cart_events_last_365d"].replace(0, np.nan)

    slice_df = (
        user_history
        .merge(customer_orders, on="user_id", how="left")
        .merge(customer_items, on="user_id", how="left")
        .merge(customer_events, on="user_id", how="left")
    )
    slice_df["cutoff_date"] = cutoff_date

    for col in slice_df.columns:
        if col in ["user_id", "cutoff_date", "churn_target", "main_category", "main_department", "main_brand", "gender", "state", "traffic_source"]:
            continue
        if pd.api.types.is_numeric_dtype(slice_df[col]):
            slice_df[col] = slice_df[col].fillna(slice_df[col].median())

    return slice_df

In [ ]:
PROJECT_DIR = Path.cwd().resolve().parent
DATA_DIR = PROJECT_DIR / "data"

data_cols = [
    "order_id", "order_item_id", "user_id", "status", "gender", "created_at", "returned_at",
    "shipped_at", "delivered_at", "sale_price", "age", "state", "city", "traffic_source",
    "category", "department", "brand", "product_id", "warehouse_name", "is_loyal"
]
event_cols = [
    "id", "user_id", "session_id", "sequence_number", "created_at", "traffic_source",
    "browser", "uri", "event_type"
]

data = pd.read_csv(DATA_DIR / "data.csv", usecols=data_cols)
events = pd.read_csv(DATA_DIR / "events.csv", usecols=event_cols)

for col in ["created_at", "returned_at", "shipped_at", "delivered_at"]:
    data[col] = parse_dt(data[col])
events["created_at"] = parse_dt(events["created_at"])

data = data.drop_duplicates(subset=["order_item_id"]).copy()
events = events.drop_duplicates(subset=["id"]).copy()

data["brand"] = data["brand"].fillna("unknown")
data["sale_price"] = data["sale_price"].clip(lower=0)
data["is_returned"] = data["returned_at"].notna().astype(int)
data["is_cancelled"] = data["status"].eq("Cancelled").astype(int)
data["is_delivered"] = data["delivered_at"].notna().astype(int)
data["created_at_naive"] = data["created_at"].dt.tz_localize(None)
data["ship_delay_days"] = (data["shipped_at"] - data["created_at"]).dt.total_seconds() / 86400
data["delivery_days"] = (data["delivered_at"] - data["shipped_at"]).dt.total_seconds() / 86400

events = events.dropna(subset=["user_id", "created_at"]).copy()
events["user_id"] = events["user_id"].astype(int)
events["created_at_naive"] = events["created_at"].dt.tz_localize(None)
events["is_product"] = events["event_type"].eq("product").astype(int)
events["is_cart"] = events["event_type"].eq("cart").astype(int)
events["is_purchase"] = events["event_type"].eq("purchase").astype(int)

orders = (
    data.groupby("order_id", as_index=False)
    .agg(
        user_id=("user_id", "first"),
        created_at_naive=("created_at_naive", "min"),
        order_value=("sale_price", "sum"),
        items_count=("order_item_id", "nunique"),
        is_loyal=("is_loyal", "max"),
    )
    .dropna(subset=["user_id", "created_at_naive"])
)

dataset = pd.concat([build_slice(cutoff, orders, data, events) for cutoff in CUTOFF_DATES], ignore_index=True)
dataset.shape

In [ ]:
train_cutoffs = pd.to_datetime(["2024-06-01", "2024-09-01", "2024-12-01"])
valid_cutoff = pd.Timestamp("2025-03-01")
test_cutoff = pd.Timestamp("2025-06-01")

drop_cols = [
    "user_id", "first_order_ever", "last_order_ever", "last_order_in_window", "first_order_in_window",
    "last_event_at", "cutoff_date", "future_orders", "churn_target"
]
feature_cols = [col for col in dataset.columns if col not in drop_cols]

train_df = dataset[dataset["cutoff_date"].isin(train_cutoffs)].copy()
valid_df = dataset[dataset["cutoff_date"] == valid_cutoff].copy()
test_df = dataset[dataset["cutoff_date"] == test_cutoff].copy()

X_train = train_df[feature_cols].copy()
y_train = train_df["churn_target"].astype(int)
X_valid = valid_df[feature_cols].copy()
y_valid = valid_df["churn_target"].astype(int)
X_test = test_df[feature_cols].copy()
y_test = test_df["churn_target"].astype(int)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [col for col in X_train.columns if col not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False)),
        ]), categorical_features),
    ]
)

model = ExtraTreesClassifier(
    n_estimators=400,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])

pipeline.fit(X_train, y_train)

valid_score = pipeline.predict_proba(X_valid)[:, 1]
test_score = pipeline.predict_proba(X_test)[:, 1]

pd.Series(
    {
        "valid_roc_auc": round(roc_auc_score(y_valid, valid_score), 4),
        "valid_pr_auc": round(average_precision_score(y_valid, valid_score), 4),
        "test_roc_auc": round(roc_auc_score(y_test, test_score), 4),
        "test_pr_auc": round(average_precision_score(y_test, test_score), 4),
    }
).to_frame("value")

## SHAP: глобальная и локальная интерпретация

С помощью `SHAP` мы отвечаем на два вопроса:

- какие признаки в среднем сильнее всего двигают модель;
- почему модель считает конкретного клиента рискованным.

In [ ]:
X_train_enc = pipeline.named_steps["preprocessor"].transform(X_train)
X_test_enc = pipeline.named_steps["preprocessor"].transform(X_test)
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()

X_train_enc = pd.DataFrame(X_train_enc, columns=feature_names, index=X_train.index)
X_test_enc = pd.DataFrame(X_test_enc, columns=feature_names, index=X_test.index)

if shap is not None:
    explainer = shap.TreeExplainer(pipeline.named_steps["model"])
    shap_sample = X_test_enc.sample(min(300, len(X_test_enc)), random_state=42)
    shap_values = explainer.shap_values(shap_sample)

    if isinstance(shap_values, list):
        shap_class_1 = shap_values[1]
    elif getattr(shap_values, "ndim", 0) == 3:
        shap_class_1 = shap_values[:, :, 1]
    else:
        shap_class_1 = shap_values

    shap.summary_plot(shap_class_1, shap_sample, max_display=15)
    shap.summary_plot(shap_class_1, shap_sample, plot_type="bar", max_display=15)


In [ ]:
top_risk_idx = pd.Series(test_score, index=X_test.index).sort_values(ascending=False).index[0]
top_risk_client = test_df.loc[top_risk_idx, ["user_id", "churn_target", "cutoff_date"] + [
    "days_since_last_order", "days_since_last_event", "return_rate_items", "orders_last_365d", "sessions_last_365d"
]]
top_risk_client

In [ ]:
if shap is not None:
    local_row = X_test_enc.loc[[top_risk_idx]]
    local_shap = explainer.shap_values(local_row)

    if isinstance(local_shap, list):
        local_class_1 = local_shap[1][0]
    elif getattr(local_shap, "ndim", 0) == 3:
        local_class_1 = local_shap[0, :, 1]
    else:
        local_class_1 = local_shap[0]

    local_contrib = (
        pd.DataFrame({
            "feature": feature_names,
            "shap_value": local_class_1,
            "abs_shap": np.abs(local_class_1),
        })
        .sort_values("abs_shap", ascending=False)
        .head(12)
    )

    plt.figure(figsize=(10, 6))
    sns.barplot(data=local_contrib.sort_values("shap_value"), x="shap_value", y="feature", palette="viridis")
    plt.title("SHAP для конкретного клиента: что сильнее всего подняло риск")
    plt.xlabel("Вклад признака в риск ухода")
    plt.ylabel("Признак")
    plt.show()


## PDP и ICE: как признаки влияют на риск

Эти графики нужны для ответа на вопрос:

- как в среднем меняется риск при росте признака;
- одинаково ли это работает для всех клиентов.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
PartialDependenceDisplay.from_estimator(
    pipeline,
    X_test,
    features=["days_since_last_order", "return_rate_items"],
    kind="average",
    ax=ax,
)
fig.suptitle("PDP: как отдельные признаки меняют риск ухода", y=1.02)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
PartialDependenceDisplay.from_estimator(
    pipeline,
    X_test.sample(min(300, len(X_test)), random_state=42),
    features=["days_since_last_order"],
    kind="both",
    ax=ax,
)
plt.title("ICE + PDP по признаку 'дней с последнего заказа'")
plt.tight_layout()
plt.show()

## LIME: локальное объяснение прогноза

`LIME` дополняет `SHAP` и показывает, какие признаки локально подтолкнули модель к решению для одного клиента.

In [ ]:
if LimeTabularExplainer is not None:
    lime_explainer = LimeTabularExplainer(
        training_data=np.asarray(X_train_enc),
        feature_names=list(feature_names),
        class_names=["Останется", "Уйдет"],
        mode="classification",
        discretize_continuous=True,
        random_state=42,
    )

    lime_exp = lime_explainer.explain_instance(
        X_test_enc.loc[top_risk_idx].values,
        pipeline.named_steps["model"].predict_proba,
        num_features=10,
    )

    lime_df = (
        pd.DataFrame(lime_exp.as_list(), columns=["feature", "weight"])
        .assign(abs_weight=lambda df: df["weight"].abs())
        .sort_values("abs_weight", ascending=False)
    )

    plt.figure(figsize=(10, 6))
    sns.barplot(data=lime_df.sort_values("weight"), x="weight", y="feature", palette="magma")
    plt.title("LIME для клиента с высоким риском")
    plt.xlabel("Локальный вклад в прогноз")
    plt.ylabel("Признак")
    plt.show()
